# セミナーII 第1回 実践：画像分類モデルを学習する

スライドの「画像分類の実践」で見た学習を、実際に動かします。題材は、犬と猫の写真を37クラスに分類する問題です（Oxford-IIIT Pet）。

**最初に**、メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選んでから、上のセルから順に実行してください。

コードを全部書ける必要はありません。スライドのどの話が、コードのどこにあるかを説明できることを目標にします。各節の見出しに、対応するスライドのページを書いています。

見出しに **[確かめよう]** が付いた節では、スライドの内容をコードで確かめます。後の節はこの節の結果を使わないので、飛ばしても最後まで動きますが、なるべく実行して確かめてください。

## 0. このノートブックの地図

### 学習に用意するもの（p88）と、この後の節

| 用意するもの | スライド | このノートブック |
| --- | --- | --- |
| 計算環境 | GPU を使う理由（p101） | 1. 準備 |
| 学習の設定 | 人間が決める学習の設定（p99） | 2. 学習の設定 |
| データセット | ① データを用意する（p90〜p94） | 3. ① データを用意する |
| モデル $F$ | CNN による画像分類（p78）、学習済みモデルから始める（p100） | 4. モデル F とパラメータ θ |
| Loss | 分類の Loss：交差エントロピー（p84） | 5. Loss と更新の方法 |
| 更新の方法 | 勾配降下法（p24）、分類モデルの学習も、同じ繰り返し（p85） | 5. Loss と更新の方法 |

そろったら、**6. ② 学習を進める**（p96〜p98）で学習し、**7. ③ 結果を確かめる**（p103、p104）、**8. 推論**（p16）と進みます。**9.** で設定を変えて実験し、**10.** が課題です。**11.**（発展）では、同じ学習を GitHub でよく見る形（`train.py`）で動かします。

### コンセプト式とコードの対応（p6、p15）

$$Y = F(X; \theta)$$

| 式 | 意味 | コード |
| --- | --- | --- |
| $X$ | 入力（画像） | `images` |
| $Y$ | 正解（クラスの番号） | `labels` |
| $F$ | 計算の形（ネットワークの形） | `model` の構造（ResNet-18） |
| $\theta$ | パラメータ（重み） | `model.parameters()` |
| $\hat{Y}$ | 予測 | `logits = model(images)` |
| Loss | 予測と正解のずれ | `loss = criterion(logits, labels)` |

## 1. 準備（計算環境。p101）

公開リポジトリから、このノートブックが使うファイル（`.py` と画像）を取得します。取得したファイルは、左の「ファイル」（フォルダのアイコン）から開けます。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/r-terasawa457/seminar2-2026.git"
REPO_REF = "main"  # 授業当日は、教材が変わらないようタグを指定する

if not Path("viz.py").exists():  # Colab では、最初はファイルが無い
    if not Path("seminar2-2026").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "seminar2-2026"],
                       check=True)
    os.chdir("seminar2-2026/lesson01")
# import で .py を見つけられるよう、作業フォルダを探す場所に加える（Colab では自動で加わらない）
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("作業フォルダ:", os.getcwd())
print(sorted(p.name for p in Path(".").iterdir()))

In [ ]:
# Colab に入っていないライブラリがあれば入れる
import importlib.util

for package in ["timm", "datasets"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

使うライブラリを読み込み、計算に使う装置（`device`）を決めます。

- `device: cuda` と表示されれば GPU で計算します（NVIDIA の GPU を使うための仕組みが CUDA）。GPU の名前（Colab では Tesla T4 など）も表示されます。
- `device: cpu` と表示されたら、GPU が選ばれていません。メニューの「ランタイム」→「ランタイムのタイプを変更」で GPU を選び、最初から実行し直してください。CPU のままでも動きますが、枚数を減らしても時間がかかります。

この後、モデルとデータは `.to(device)` で、この装置に送ってから計算します。

In [ ]:
import copy
import json
import time
from functools import partial

import numpy as np
import timm
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from torch.utils.data import DataLoader
from torchvision import transforms

import viz  # 表示用の関数（viz.py）。学習の仕組みとは関係しない

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

## 2. 学習の設定（p99）

$\theta$ は機械が求めますが、学習の設定（ハイパーパラメータ）は人間が決めます。ここで決めるのは本編（3.〜8.）の設定です。設定を変えた実験は、9. で行います。

In [ ]:
MODEL_NAME = "resnet18.a1_in1k"  # モデル F（timm の名前）
PRETRAINED = True                # 学習済みの重みから始めるか（p100）
EPOCHS = 3                       # データ全体を何周するか（p97）
BATCH_SIZE = 64                  # 1回の更新に使う枚数（p98）
LEARNING_RATE = 1e-3             # 学習率（p25）
SEED = 0                         # 乱数の種。同じなら、データの分け方が同じになる（学習の結果は、実行のたびに少し変わる）

IMAGE_DIR = None  # 自分で集めた画像で学習するときは、フォルダ名を書く（10. 課題の発展）

# データセットの取得元。授業の途中で内容が変わらないよう、リビジョンを固定している
DATASET_ID = "timm/oxford-iiit-pet"
DATASET_REVISION = "089695c834a7deb60505b7cc506672db1c31a6aa"

# GPU が無いときは、枚数を減らして早く終わらせる（None は全部）
MAX_TRAIN_IMAGES = None if device == "cuda" else 600
MAX_TEST_IMAGES = None if device == "cuda" else 300
NUM_WORKERS = 2 if device == "cuda" and os.name != "nt" else 0  # データを読む並列数

torch.manual_seed(SEED)

## 3. ① データを用意する（p90〜p94）

### 3-1. データセット：Oxford-IIIT Pet（p90）

犬と猫の写真に、品種名の正解ラベルが付いたデータセットです。Hugging Face Hub から取得します（約800MB、初回のみ）。ログインは不要で、`HF_TOKEN` に関する警告が表示されても無視してかまいません。

In [ ]:
if IMAGE_DIR is None:
    dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION)
else:
    # フォルダ名がクラス名になる。test が無いので、2割を test として取り分ける
    images = load_dataset("imagefolder", data_dir=IMAGE_DIR)["train"]
    dataset = images.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
print(dataset)

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)
print("クラスの数:", num_classes)
print(class_names)

In [ ]:
# スライドと同じ8クラスの見本。括弧の中はクラスの番号
viz.show_samples(dataset["train"], class_names,
                 ["Persian", "Siamese", "Sphynx", "British_Shorthair", "shiba_inu", "pug", "beagle", "samoyed"])

# この後の説明で例として使う1件（スライドと同じ柴犬の写真）
sample = dataset["train"][49] if IMAGE_DIR is None else dataset["train"][0]

#### [確かめよう] データ1件：画像と正解ラベルの組（p91）

データセットの1件は、入力の画像と、正解のラベル（クラスの番号）の組です（p14）。クラス名は 0〜36 の番号で扱います。

In [ ]:
print(sample.keys())
print("正解ラベル:", sample["label"], "→", class_names[sample["label"]])
viz.show_example(sample, class_names)

In [ ]:
# 画像は画素値の配列（p47）。高さ × 幅 × 3（赤、緑、青）の、0〜255 の整数
pixels = np.array(sample["image"].convert("RGB"))
print("形:", pixels.shape, " 型:", pixels.dtype)
h, w = pixels.shape[:2]
print("中央の画素の (赤, 緑, 青):", pixels[h // 2, w // 2])

### 3-2. データを3つに分ける（p92）

学習に使っていないデータで評価するために、データを3つに分けます。Oxford-IIIT Pet には train と test しかないので、train の2割を validation に回します。

| データ | 役割 | たとえ |
| --- | --- | --- |
| train | 重みの更新に使う | 練習問題 |
| validation | 設定の判断、重みの選択 | 模試 |
| test | 最後に1回だけ測る | 本番の試験 |

In [ ]:
# 各クラスの比率がそろうように分ける（stratify）
split = dataset["train"].train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
train_data, val_data, test_data = split["train"], split["test"], dataset["test"]

if MAX_TRAIN_IMAGES is not None:
    train_data = train_data.shuffle(seed=SEED).select(range(min(MAX_TRAIN_IMAGES, len(train_data))))
    val_data = val_data.shuffle(seed=SEED).select(range(min(MAX_TRAIN_IMAGES // 4, len(val_data))))
if MAX_TEST_IMAGES is not None:
    test_data = test_data.shuffle(seed=SEED).select(range(min(MAX_TEST_IMAGES, len(test_data))))

print("train      :", len(train_data))
print("validation :", len(val_data))
print("test       :", len(test_data))

### 3-3. 前処理とデータ拡張（p93、p94）

**前処理**（p93）：大きさと値の範囲をそろえてから、モデルに入れます。

1. **大きさをそろえる**：縮小して、中央を 224×224 で切り抜く
2. **値の範囲をそろえる（正規化）**：値 ＝（画素値 ÷ 255 − 平均）÷ 標準偏差。チャンネルごとに計算する

学習済みモデルを使うときは、そのモデルを学習したとき（ImageNet）と同じ前処理にします。

**データ拡張**（p94）：学習のときだけ、毎回ランダムに「一部を切り抜いて拡大（面積の50〜100%）」「左右を反転」します。同じ画像を丸覚えしにくくなります。評価（validation、test）では使いません。

下の関数は、学習用（データ拡張つき）と評価用の2つを作ります。

In [ ]:
IMAGE_SIZE = 224
# 学習済みモデルを ImageNet で学習したときの平均と標準偏差（p93）
MEAN = IMAGENET_DEFAULT_MEAN
STD = IMAGENET_DEFAULT_STD


def build_transforms(image_size=IMAGE_SIZE, mean=MEAN, std=STD):
    """前処理（p93）とデータ拡張（p94）を作る。学習用と評価用の2つを返す"""
    normalize = transforms.Normalize(mean, std)
    # 評価用：大きさをそろえて（縮小して中央を切り抜く）、値の範囲をそろえる
    eval_transform = transforms.Compose([
        transforms.Resize(int(image_size * 1.15)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        normalize,
    ])
    # 学習用：毎回ランダムに一部を切り抜いて拡大し、左右を反転する
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.5, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    return train_transform, eval_transform


train_transform, eval_transform = build_transforms()
print("学習用:", train_transform)
print("評価用:", eval_transform)

#### [確かめよう] 前処理の前と後（p93）

前処理の後は、大きさが 224×224 になり、値は 0 をはさんだ範囲になります。スライドの数値例（赤の値が 191 の画素）も計算してみます。

In [ ]:
print("平均      (赤, 緑, 青):", MEAN)
print("標準偏差  (赤, 緑, 青):", STD)
x = eval_transform(sample["image"])
print("前処理の後の形 (チャンネル, 高さ, 幅):", tuple(x.shape))
print("値の範囲:", round(x.min().item(), 2), "〜", round(x.max().item(), 2))
print("(191 ÷ 255 − 0.485) ÷ 0.229 =", round((191 / 255 - MEAN[0]) / STD[0], 2))

viz.show_preprocess(sample["image"], eval_transform)

#### [確かめよう] データ拡張をかけた画像（p94）

同じ画像に、学習用の変換を何回かかけて並べます。もう一度実行すると、毎回違う画像になります。

In [ ]:
viz.show_augmentations(sample["image"], train_transform)

### 3-4. mini-batch：少しずつ取り出す（p98）

学習では、画像を1枚ずつではなく、`BATCH_SIZE` 枚（64枚）をまとめて計算します。64枚の画像（どれも 3×224×224 の配列。p47）を重ねて、1つの大きな配列（64×3×224×224）にし、GPU で一度に計算します（p101）。このような多次元の配列を、PyTorch では**テンソル**と呼びます。正解も、64枚分のクラスの番号を並べた1つのテンソルにします。

`DataLoader` は、画像に前処理をかけ、`BATCH_SIZE` 枚ずつ束ねて取り出します。train は毎 epoch、順番を混ぜます（`shuffle=True`）。

In [ ]:
def _apply(batch, transform):
    return {
        "image": [transform(img.convert("RGB")) for img in batch["image"]],
        "label": batch["label"],
    }


def make_loader(data, transform, batch_size, shuffle, num_workers=0):
    """画像をテンソルに変換し、batch_size 枚ずつ束ねて取り出す（p98）"""
    return DataLoader(data.with_transform(partial(_apply, transform=transform)),
                      batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)


train_loader = make_loader(train_data, train_transform, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = make_loader(val_data, eval_transform, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_loader(test_data, eval_transform, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
batch = next(iter(train_loader))
print("入力 X：画像を64枚まとめたテンソル")
print("  形 (枚数, チャンネル, 高さ, 幅):", tuple(batch["image"].shape))
print("正解 Y：64枚それぞれの正解のクラスの番号を並べたテンソル")
print("  形 (枚数,):", tuple(batch["label"].shape))
print("  先頭の8枚:", ", ".join(f"{i}（{class_names[i]}）" for i in batch["label"][:8].tolist()))
print()
# mini-batch 1つで θ を1回更新する（p98）
print(f"1 epoch（train {len(train_data)} 枚を1周）で θ を更新する回数:", len(train_loader),
      f"回（{len(train_data)} 枚 ÷ {BATCH_SIZE} 枚。割り切れない分も1回）")
print(f"学習全体（{EPOCHS} epoch）で θ を更新する回数:", EPOCHS * len(train_loader), "回")
viz.show_batch(batch, class_names)

## 4. モデル F とパラメータ θ（p78、p100）

### 4-1. ResNet-18 を作る（p78、p100）

`timm` というライブラリで、CNN の一つ ResNet-18 を作ります。形はスライド p78 の図のとおりで、**特徴抽出器**（畳み込み・ReLU・Pooling の繰り返し）と**分類ヘッド**（平均 → 全結合）からできています。

`pretrained=True` にすると、ImageNet で学習済みの重みが $\theta$ に読み込まれます。`num_classes` を指定すると、1000クラス用の分類ヘッドが37クラス用に付け替えられます。特徴抽出器の重みは引き継ぎ、新しい分類ヘッドの重みはランダムな値から始まります（転移学習。p100）。

In [ ]:
def build_model(model_name, num_classes, pretrained=True):
    """F（ネットワークの形）を作り、θ（重み）に学習済みの値を入れる

    num_classes を指定すると、分類ヘッドが num_classes クラス用に付け替えられる（p100）。
    使えるモデルの名前は timm.list_models(pretrained=True) で調べられる。
    """
    return timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)


model = build_model(MODEL_NAME, num_classes, pretrained=PRETRAINED).to(device)

#### [確かめよう] 層を通るごとの形（p62〜p66）

画像を1枚入れて、層を通るごとに出力の形がどう変わるかを見ます。下のセルの出力は、左から「層の名前」「その層の出力の形（チャンネル, 高さ, 幅）」「層の種類」です。

| 層の名前 | 役割 | スライド p78 の図 |
| --- | --- | --- |
| `conv1` | 畳み込み層（7×7 のカーネルが64枚。p60） | 図では省略（224×224 と 56×56 の間） |
| `bn1` | 正規化（Batch Normalization）。値の範囲を整える層（スライドでは名前だけ。p72） | 図では省略 |
| `act1` | ReLU（p63） | 図では省略 |
| `maxpool` | Pooling 層。大きさを半分にする（p64） | 図では省略 |
| `layer1`〜`layer4` | 畳み込み・正規化・ReLU の積み重ね（ショートカット接続を含む。p40）。だんだん小さく、チャンネルは多くなる（p66、p72） | 56×56 → 28×28 → 14×14 → 7×7 |
| `global_pool` | Global Average Pooling。7×7 の特徴マップを、チャンネルごとに平均する。512チャンネルなので、512次元のベクトルになる。出力の形が `(512,)` なのはこのため（p80） | 平均 |
| `fc` | 全結合層。512次元のベクトルから、37クラスのスコアを出す（p82） | 全結合 512 → 37 |

`conv1`〜`layer4` が特徴抽出器、`global_pool` と `fc` が分類ヘッドです。

In [ ]:
viz.print_feature_shapes(model, device)

# 中身をすべて表示する（長いので、興味があれば # を外す）
# print(model)

#### [確かめよう] θ に含まれるパラメータの数（p78）

$\theta$ はスカラーではなく、カーネルの値や全結合層の重みとバイアスなど、たくさんのパラメータをまとめて表したものです。コードでは `model.parameters()` に入っています。学習では、これらのパラメータをすべて少しずつ動かします。スライド p78 の「重みの合計は約1100万個」を確かめます。

In [ ]:
n_all = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.get_classifier().parameters())
print(f"θ に含まれるパラメータの数（全体）: {n_all:,} 個")
print(f"  特徴抽出器                      : {n_all - n_head:,} 個")
print(f"  分類ヘッド                      : {n_head:,} 個")
print()
print("最初の畳み込み層の重みの形 (枚数, 入力チャンネル, 高さ, 幅):", tuple(model.conv1.weight.shape))
print("分類ヘッドの重みの形 (クラス数, 特徴の数):", tuple(model.fc.weight.shape))

#### [確かめよう] ImageNet で学習したカーネル（p60、p100）

4-1 で読み込んだ重みのうち、最初の畳み込み層のカーネル64枚を表示します。スライド p60 と同じものです。

このノートブックの学習（6.）はまだしていません。これは、別の大きなデータセット ImageNet（約128万枚、1000クラス）で学習したときに求まった値です。今回は、ここから学習を始めます（転移学習。p100）。人間が値を決めたわけではないのに、境界や色に反応するものが見えます（真っ黒なものは、値がすべて0）。

`PRETRAINED = False` にすると、ImageNet で学習した値を読み込まないので、ランダムな値になります。

In [ ]:
viz.show_kernels(model)

#### [確かめよう] 分類ヘッドの付け替え（p100）

付け替える前と後の分類ヘッドを比べます。出力の数（`out_features`）が 1000 から 37 に変わっています。

In [ ]:
original = timm.create_model(MODEL_NAME, pretrained=False)  # 付け替える前の形（重みは読み込まない）
print("元の分類ヘッド    :", original.get_classifier())
print("今回の分類ヘッド  :", model.get_classifier())
del original

#### [確かめよう] 学習の前に予測してみる（p82）

分類ヘッドが出す37個のスコア（logits）を、softmax で合計1の確率に変換します。分類ヘッドがまだランダムなので、ほとんど当てずっぽうです（赤が正解のクラス）。

In [ ]:
model.eval()
with torch.no_grad():
    x = eval_transform(sample["image"]).unsqueeze(0).to(device)  # 1枚だけの mini-batch にする
    logits = model(x)                                            # 37個のスコア
probs = logits.softmax(dim=1)[0].cpu()                           # 合計1の確率に変換する

print("確率の合計:", round(probs.sum().item(), 4))
print("正解のクラスの確率:", round(probs[sample["label"]].item(), 4),
      f" （当てずっぽうなら 1/{num_classes} ≒ {1 / num_classes:.4f}）")
viz.show_probs(probs, class_names, true_label=sample["label"])

## 5. Loss と更新の方法（p84、p85）

### 5-1. 交差エントロピーと AdamW（p84、p24）

**Loss**：分類では交差エントロピーを使います。正解のクラスの確率 $p$ を使って、Loss ＝ $-\log p$ です（p84）。正解の確率が1に近いほど0に近く、0に近いほど大きくなります。

**更新の方法**：勾配降下法（p24）では、$\theta$ を「勾配に応じて、Loss が小さくなる方向に少し動かす」ことを繰り返します。動かす幅を決めるのが学習率です（p25）。ここでは、勾配降下法の改良版の一つ AdamW を使います。`model.parameters()`（$\theta$ の全部）を渡して、更新する対象にします。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(criterion)
print(optimizer)

#### [確かめよう] 交差エントロピーは −log p（p84）

例の1件で、`criterion` が計算した Loss と、$-\log p$ を比べます。

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(eval_transform(sample["image"]).unsqueeze(0).to(device))  # Ŷ = F(X; θ)
label = torch.tensor([sample["label"]], device=device)                        # Y
p = logits.softmax(dim=1)[0, sample["label"]]
print("criterion で計算した Loss:", round(criterion(logits, label).item(), 4))
print("−log(正解の確率)         :", round(-torch.log(p).item(), 4))

#### [確かめよう] 1回だけ更新してみる（p15、p85）

3-4 で取り出した mini-batch 1つで、学習の繰り返しを1回だけ手で実行します。$\theta$ が少し動き、同じ mini-batch の Loss が下がることを確かめます。

最後に $\theta$ を元に戻すので、このセルを実行してもしなくても、6. の結果は変わりません。

In [ ]:
saved_model = copy.deepcopy(model.state_dict())          # 元に戻すために控えておく
saved_optimizer = copy.deepcopy(optimizer.state_dict())
images = batch["image"].to(device)  # 入力 X
labels = batch["label"].to(device)  # 正解 Y


def batch_loss():
    model.eval()
    with torch.no_grad():
        return criterion(model(images), labels).item()


before_loss, before_w = batch_loss(), model.fc.weight[0, :3].tolist()

model.train()
logits = model(images)            # 予測を計算する：Ŷ = F(X; θ)
loss = criterion(logits, labels)  # Loss を計算する
optimizer.zero_grad()             # （前回の勾配を消す）
loss.backward()                   # 勾配を計算する（誤差逆伝播）
optimizer.step()                  # θ を更新する

print("θ の一部（分類ヘッドの重み3個）")
print("  更新前:", np.round(before_w, 4))
print("  更新後:", np.round(model.fc.weight[0, :3].tolist(), 4))
print(f"同じ mini-batch の Loss: {before_loss:.3f} → {batch_loss():.3f}")

model.load_state_dict(saved_model)  # θ を元に戻す
optimizer.load_state_dict(saved_optimizer)

## 6. ② 学習を進める（p96〜p98）

mini-batch ごとに学習の繰り返しを1回行い、train 全部を1周すると 1 epoch です。それを `EPOCHS` 回繰り返し、1周ごとに validation で正解率を測って、一番良かった時点の重みを残します（p96）。

`train_one_epoch` の中の4行が、スライドの学習の繰り返しです（p85）。

| 学習の繰り返し | コード |
| --- | --- |
| 予測を計算する $\hat{Y} = F(X; \theta)$ | `logits = model(images)` |
| Loss を計算する | `loss = criterion(logits, labels)` |
| 勾配を計算する（誤差逆伝播） | `loss.backward()` |
| $\theta$ を更新する | `optimizer.step()` |

`evaluate` は、$\theta$ を更新せずに Loss と正解率を測ります（validation と test で使う）。`model.train()` と `model.eval()` は、データ拡張とは別に、学習と評価で動きが変わる層（`bn1` など）を切り替えます。

`fit` は、epoch の繰り返し（p96 の外側の繰り返し）です。9. の実験でも、この関数を使います。

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """train を1周（1 epoch）学習する。mini-batch ごとに θ を1回更新する（p96〜98）"""
    model.train()
    total_loss, correct, count = 0.0, 0, 0
    for batch in loader:
        images = batch["image"].to(device)   # 入力 X
        labels = batch["label"].to(device)   # 正解 Y

        logits = model(images)               # 予測を計算する：Ŷ = F(X; θ)
        loss = criterion(logits, labels)     # Loss を計算する
        optimizer.zero_grad()                # （前回の勾配を消す）
        loss.backward()                      # 勾配を計算する（誤差逆伝播）
        optimizer.step()                     # θ を更新する

        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += len(labels)
    return total_loss / count, correct / count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """θ を更新せずに、Loss と正解率を測る（validation、test。p92）"""
    model.eval()
    total_loss, count = 0.0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * len(labels)
        count += len(labels)
        all_preds.append(logits.argmax(dim=1).cpu())
        all_labels.append(labels.cpu())
    preds, labels = torch.cat(all_preds), torch.cat(all_labels)
    return total_loss / count, (preds == labels).float().mean().item(), preds, labels


def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    """epochs 回学習する。1周ごとに validation で測り、一番良かった時点の重みを残す（p96）"""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(f"epoch {epoch}/{epochs}  train loss {train_loss:.3f} acc {train_acc:.3f}"
              f"  |  val loss {val_loss:.3f} acc {val_acc:.3f}  |  {time.time() - start:.0f}s")
        # validation の正解率が一番良い時点の重みを残す
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)  # 一番良かった時点の重みに戻す
    return history, best_acc

In [ ]:
history, best_acc = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS)
print(f"best val acc {best_acc:.3f}")

## 7. ③ 結果を確かめる（p103、p104）

### 7-1. 学習曲線（p103）

train の Loss は下がり続けます。validation の Loss が途中から上がり始めたら、過学習のサインです。

In [ ]:
viz.plot_history(history)

### 7-2. test で測る（p104）

学習にも設定の判断にも使っていない test で、最後に1回だけ測ります。数字だけで終わらせず、どんな間違い方をしているかも見ます。下の図では、間違えた画像の横に、正解のクラスと予測したクラスの見本を並べます。

同じ設定でも、実行するたびに正解率は数ポイント変わります（例：0.80〜0.83）。`SEED` を固定していても、GPU の計算が毎回まったく同じにはならないことや、どのセルを実行したかで乱数の使われ方が変わることが原因です。

In [ ]:
test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion, device)
print(f"train      acc {history['train_acc'][-1]:.3f}  （{EPOCHS} epoch 目）")
print(f"validation acc {best_acc:.3f}  （最良）")
print(f"test       acc {test_acc:.3f}  （{len(labels)} 枚）")
print(f"参考：当てずっぽうなら 1/{num_classes} ≒ {1 / num_classes:.3f}")
print()
viz.print_class_accuracy(preds, labels, class_names)
print()
print("間違いが多かった組み合わせ（正解 -> 予測）")
viz.print_confusions(preds, labels, class_names)

In [ ]:
# 間違いが多かった組み合わせから1枚ずつ。右の2列は、正解のクラスと予測したクラスの見本（train から）
viz.show_mistakes(test_data, preds, labels, class_names, reference=train_data)

## 8. 推論（p16）

### 8-1. 学習したモデルで予測する（p16、p82）

学習が終われば、使い方は古典的手法と同じで、画像を入れると答えが出ます（$\theta$ は学習で求めた値に固定）。`predict` は、37個のスコアを softmax で確率に変換して返します。test の画像と猫の画像で試します。

下の段には、上位のクラスの見本の画像（train から1枚ずつ）を並べます。品種に詳しくなくても、何と似ていると判断したのかを比べられます。

In [ ]:
@torch.no_grad()
def predict(model, image, transform, device):
    """1枚の画像（PIL）を入力し、各クラスの確率を返す（推論。p16、p82）"""
    model.eval()
    x = transform(image.convert("RGB")).unsqueeze(0).to(device)  # 1枚だけの mini-batch にする
    return model(x).softmax(dim=1)[0].cpu()


probs = predict(model, test_data[0]["image"], eval_transform, device)
print("正解:", class_names[test_data[0]["label"]])
viz.show_prediction(test_data[0]["image"], probs, class_names, reference=train_data)

cat = Image.open("assets/chelsea.png")
probs = predict(model, cat, eval_transform, device)
viz.show_prediction(cat, probs, class_names, reference=train_data)

#### [確かめよう] 学習していないものを入れると（p105）

モデルは、学習した37クラスのどれかでしか答えられません。確率は合計が1になるように変換しただけなので、高くても正しいとは限りません。

In [ ]:
coffee = Image.open("assets/coffee.png")
probs = predict(model, coffee, eval_transform, device)
viz.show_prediction(coffee, probs, class_names, reference=train_data)

#### [確かめよう] 層ごとの特徴マップ（p71）

推論のとき、モデルの中で画像がどう変わっていくかを見ます。

- **上の5段（特徴抽出器）**：各層の特徴マップから、平均的に強く反応したものを6枚ずつ表示します。深い層ほど小さくなり、1マスが元の画像の広い範囲に対応します（p66）。
- **下の3段（分類ヘッド。p80〜p82）**：
  1. `global_pool`（Global Average Pooling）：`layer4` の特徴マップ（7×7、512枚）を1枚ずつ平均した、512次元の特徴ベクトル
  2. `fc`：それを全結合層に通した、37クラスのスコア（logits）。負の値もある
  3. `softmax`：スコアを、全部足すと1になるように変換した確率。スコアの大小の順番は変わらない

  横軸はチャンネルの番号とクラスの番号で、確率の高い3クラスに名前を付けています。

In [ ]:
viz.show_feature_maps(model, cat, eval_transform, device, class_names)

### 8-2. 自分の画像を入れる

**自分で撮った画像**を入れてみましょう。`USE_UPLOAD = True` にして実行すると、アップロードのボタンが表示されます（Colab のみ）。犬や猫以外（タヌキ、ぬいぐるみ、自動車など）も試してください。

In [ ]:
USE_UPLOAD = False

if USE_UPLOAD:
    from google.colab import files

    for name in files.upload():
        image = Image.open(name)
        viz.show_prediction(image, predict(model, image, eval_transform, device), class_names, reference=train_data)

## 9. 実験：設定やコードを変えて比べる（p99）

設定を変えて学習し直し、結果を比べます。`run_experiment` は、ここまでのセルで定義した関数（`build_transforms`、`make_loader`、`build_model`、`fit`、`evaluate`）を使って、データの用意から test までを通して実行し、結果を表（`results`）に加えます。

- 引数で指定しなかった設定は、2. の値のままです。
- 1回あたり、GPU（T4）で約1.5分かかります（3 epoch の場合）。
- 本編のモデル（6. で学習し、8. で使ったもの）は上書きしません。

同じ設定でも、test の正解率は実行のたびに数ポイント変わります（7-2）。それより小さい差は、設定の違いによるものとは言えません。

In [ ]:
# 本編（6.、7-2）の結果を、比べる基準として最初に入れておく
results = [{"note": "本編（6.）", "changes": "baseline", "model": MODEL_NAME, "pretrained": PRETRAINED,
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
            "best_val_acc": round(best_acc, 3), "test_acc": round(test_acc, 3), "history": history}]


def run_experiment(note, model_name=MODEL_NAME, pretrained=PRETRAINED, epochs=EPOCHS,
                   batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE):
    """設定を変えて、データの用意から test まで通して実行し、結果（学習の記録も）を results に加える"""
    # 2. の設定から変えたものを、図の見出し用に英数字で記録する
    settings = {"model_name": model_name, "pretrained": pretrained, "epochs": epochs,
                "batch_size": batch_size, "learning_rate": learning_rate}
    defaults = {"model_name": MODEL_NAME, "pretrained": PRETRAINED, "epochs": EPOCHS,
                "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE}
    changes = ", ".join(f"{k}={v}" for k, v in settings.items() if v != defaults[k]) or "same as baseline"

    torch.manual_seed(SEED)
    train_transform, eval_transform = build_transforms()  # 3-3（セルを直して実行し直すと、ここにも反映される）
    train_loader = make_loader(train_data, train_transform, batch_size, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = make_loader(val_data, eval_transform, batch_size, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = make_loader(test_data, eval_transform, batch_size, shuffle=False, num_workers=NUM_WORKERS)
    model = build_model(model_name, num_classes, pretrained=pretrained).to(device)  # 4-1
    criterion = nn.CrossEntropyLoss()                                               # 5-1
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history, best_acc = fit(model, train_loader, val_loader, criterion, optimizer, device, epochs)  # 6
    _, test_acc, _, _ = evaluate(model, test_loader, criterion, device)                          # 7-2
    print(f"best val acc {best_acc:.3f}  |  test acc {test_acc:.3f}")
    results.append({"note": note, "changes": changes, "model": model_name, "pretrained": pretrained,
                    "epochs": epochs, "batch_size": batch_size, "lr": learning_rate,
                    "best_val_acc": round(best_acc, 3), "test_acc": round(test_acc, 3), "history": history})

最初の例として、3つの実験をします。それぞれ、本編とどこが違う結果になるかを予想してから実行しましょう。

In [ ]:
# 例1：epoch 数を増やす（p97、p103）。約2.5分
run_experiment("epoch を8に", epochs=8)

In [ ]:
# 例2：学習率を100倍にする（p25）
run_experiment("学習率を大きく", learning_rate=0.1)

In [ ]:
# 例3：学習済みの重みを使わない（p100）
run_experiment("学習済みの重みなし", pretrained=False)

In [ ]:
# 結果を比べる。図の見出しの #番号 は表の番号と同じ。見出しには、本編から変えた設定を表示する
viz.show_results(results)

### やってみよう

例のセルをコピーして引数を変え、実行しましょう。最後に `viz.show_results(results)` を実行すると、それまでの実験がすべて表と図に並びます。`note` には、何を変えたかを書いておきます。

引数で指定しなかった設定は、次の値（2. の設定）のままです。

| 引数 | 既定の値 | 意味 |
| --- | --- | --- |
| `model_name` | `"resnet18.a1_in1k"` | モデル |
| `pretrained` | `True` | 学習済みの重みから始めるか |
| `epochs` | `3` | データ全体を何周するか |
| `batch_size` | `64` | 1回の更新に使う枚数 |
| `learning_rate` | `0.001` | 学習率 |

1. 学習率を小さくする：`learning_rate=1e-5`（p25）
2. epoch 数を減らす：`epochs=1`（p97）
3. 学習済みの重みを使わずに、epoch 数を増やす：`pretrained=False, epochs=8`（p100）
4. batch サイズを変える：`batch_size=16`（p98）
5. モデルを変える：`model_name="resnet50.a1_in1k"`（大きいモデル。時間は約2倍）
6. **コードを直す**：3-3 のセルの `build_transforms` から `RandomHorizontalFlip` の行を消し、**3-3 のセルを実行し直してから**、`run_experiment("左右反転なし")` を実行する（p94）。終わったら、行を戻して 3-3 のセルをもう一度実行しておく

## 10. 課題（p108）

高い精度を競う課題ではありません。実験と考察の結果を PDF にまとめて提出してください。

**確認項目**

- [ ] 最後まで実行できた
- [ ] 学習に用意するもの（データ、モデル、Loss、更新の方法、設定）と学習の繰り返しの4行を、コードを指して説明できる
- [ ] train / validation / test の違いを説明できる
- [ ] 自分の画像で推論した
- [ ] 誤分類を1例以上取り上げ、原因を考えた（例：タヌキの画像を入れたら、猫の品種と判定された。耳の形が似ているからではないか）

**実験（1つ以上）**：9. で設定を1つ変えて、結果を比べてください（学習率、epoch 数、学習済みの有無、batch サイズ、モデル、データ拡張）。

同じ設定でも、正解率は実行のたびに数ポイント変わります（7-2）。差が数ポイント以内なら、偶然の範囲かもしれないことに注意して考察してください。同じ設定で2回実行して、どのくらい変わるかを確かめるのもよい方法です。

**発展**：自分で集めた画像で分類器を作る。クラスごとにフォルダを分けて画像を入れ（例：`my_images/cat/`、`my_images/dog/`）、2. で `IMAGE_DIR = "my_images"` にして最初から実行します。11. の `train.py` では `--image-dir my_images` です。

## 11. 発展：同じ学習を train.py で動かす

研究では、GitHub で公開されたコードを動かし、読んで、直して使うことがよくあります。そうしたコードは、ノートブックではなく、役割ごとのファイルに分かれていて、コマンドで実行する形が多いです。

1. で取得したリポジトリの `seminar2-2026/lesson01/` フォルダ（今の作業フォルダ）の `.py` には、ここまでのセルと同じ処理が入っています。左の「ファイル」（フォルダのアイコン）から `seminar2-2026` → `lesson01` と開き、ファイルをダブルクリックすると中身を見られます。見比べてみましょう。

| ファイル | 中身 | このノートブック |
| --- | --- | --- |
| `data.py` | `load_splits`、`build_transforms`、`make_loader` | 3. ① データを用意する |
| `models.py` | `build_model` | 4-1 |
| `engine.py` | `train_one_epoch`、`evaluate`、`fit`、`predict` | 6、8-1 |
| `train.py` | 設定（`argparse`）、Loss、更新の方法、学習、test、保存 | 2、5-1、6、7-2 |
| `predict.py` | 保存した重みで推論する | 8-1 |
| `viz.py` | 表示用 | （各節の図） |

学習の設定は、2. のセルの代わりに、コマンドの引数で渡します。使える引数は `--help` で分かります。

In [ ]:
!python train.py --help

In [ ]:
# GPU が無いときは枚数を減らす
options = f"--num-workers {NUM_WORKERS}" + ("" if device == "cuda" else " --max-train 600 --max-test 300")
!python train.py --epochs 3 {options}

In [ ]:
# 保存した重み（outputs/best.pt）で推論する
!python predict.py outputs/best.pt assets/chelsea.png assets/coffee.png

In [ ]:
# 保存した学習の記録（outputs/history.json）を表示する
saved = json.loads(Path("outputs/history.json").read_text(encoding="utf-8"))
print(f"best val acc {saved['best_val_acc']:.3f}  |  test acc {saved['test_acc']:.3f}")
viz.plot_history(saved)

### やってみよう

`--out` で保存先を変えると、結果を並べて残せます。

1. 学習率を変える：`!python train.py --lr 0.1 --out outputs/lr0.1`（p25）
2. 学習済みの重みを使わない：`!python train.py --no-pretrained --out outputs/scratch`（p100）
3. モデルを変える：`!python train.py --model resnet50.a1_in1k --out outputs/resnet50`
4. **ファイルを直す**：`data.py` の `build_transforms` から `RandomHorizontalFlip` の行を消して保存し、`train.py` を実行して結果を比べる（p94）

**注意**：ノートブックのセルと `.py` のコードは別々のものです。`.py` を直しても、ノートブックのセル（3-3 の `build_transforms` など）には反映されません。逆に、セルを直しても `.py` には反映されません。`train.py` は実行のたびに `.py` を読み直すので、`.py` を直したら、そのまま `train.py` を実行し直せば反映されます。